In [5]:
import pandas as pd
import sqlite3

In [6]:
conn = sqlite3.connect('data/bible.eng.db')

In [3]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(query, conn)
tables

,name
0,_prisma_migrations
1,Translation
2,Book
3,Chapter
4,ChapterVerse
5,ChapterFootnote
6,ChapterAudioUrl
7,Commentary
8,CommentaryBook
9,CommentaryChapter


In [7]:
query = """
SELECT 
    t.shortName AS translation,
    t.englishName AS translation_name,
    cv.text
FROM ChapterVerse cv
JOIN Book b ON cv.bookId = b.id AND cv.translationId = b.translationId
JOIN Translation t ON cv.translationId = t.id
WHERE b.commonName = 'Romans'
  AND cv.chapterNumber = 1
  AND cv.number = 17
ORDER BY t.shortName;
"""
romans_1_17 = pd.read_sql_query(query, conn)
pd.set_option('display.max_colwidth', None)
romans_1_17

,translation,translation_name,text
0,ASV,American Standard Version (1901),"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith."
1,ASVBT,ASV Byzantine Text,"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith."
2,BBE,Bible in Basic English,"For in it there is the revelation of the righteousness of God from faith to faith: as it is said in the holy Writings, The man who does righteousness will be living by his faith."
3,BSB,Berean Standard Bible,"For the gospel reveals the righteousness of God that comes by faith from start to finish, just as it is written: “The righteous will live by faith.”"
4,DBY,Darby Translation,"for righteousness of God is revealed therein, on the principle of faith, to faith: according as it is written, But the just shall live by faith."
5,DRA,Douay-Rheims 1899,"For the justice of God is revealed therein, from faith unto faith, as it is written: The just man liveth by faith."
6,EMTV,English Majority Text Version,"For in it the righteousness of God is revealed from faith to faith; as it is written, “The just shall live by faith.”"
7,FBV,Free Bible Version,"For in the good news God is revealed as good and right, trustworthy from start to finish. As Scripture says, “Those who are right with God live by trusting him.”"
8,GNB,Noyes Bible,"For therein is revealed the righteousness which is of God from faith to faith; as it is written, “But the righteous shall live by faith.”"
9,GNV,Geneva Bible 1599,"For by it the righteousnesse of God is reueiled from faith to faith: as it is written, The iust shall liue by faith."


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# Load model
model = SentenceTransformer("google/embeddinggemma-300m")

# Get all verse texts
texts = romans_1_17['text'].tolist()

# Encode all texts as documents
embeddings = model.encode_document(texts)

# Find the KJAV index
kjav_idx = romans_1_17[romans_1_17['translation'] == 'KJAV'].index[0]
kjav_embedding = embeddings[kjav_idx]

# Compute cosine similarity between KJAV and all others
similarities = model.similarity(kjav_embedding, embeddings).squeeze()

# Build results dataframe
similarity_df = romans_1_17[['translation', 'translation_name', 'text']].copy()
similarity_df['similarity_to_KJAV'] = similarities.numpy()
similarity_df = similarity_df.sort_values('similarity_to_KJAV', ascending=False)
similarity_df

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

,translation,translation_name,text,similarity_to_KJAV
11,KJVA,King James Version + Apocrypha,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
12,KJVCP,KJV Cambridge Paragraph,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
10,KJAV,King James Version,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
16,NWB,Webster Bible,"For in this is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",0.988846
6,EMTV,English Majority Text Version,"For in it the righteousness of God is revealed from faith to faith; as it is written, “The just shall live by faith.”",0.986585
0,ASV,American Standard Version (1901),"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith.",0.972933
1,ASVBT,ASV Byzantine Text,"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith.",0.972933
8,GNB,Noyes Bible,"For therein is revealed the righteousness which is of God from faith to faith; as it is written, “But the righteous shall live by faith.”",0.971429
18,RVA,Revised Version,"For therein is revealed a righteousness of God by faith unto faith: as it is written, But the righteous shall live by faith.",0.967390
29,WEBBE,World English Bible British Edition with Deuterocanon,"For in it is revealed God’s righteousness from faith to faith. As it is written, “But the righteous shall live by faith.”",0.967063


: 